In [1]:
from pathlib import Path
import os

# If we're inside the notebooks/ folder, move up to the project root
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(str(cwd.parent))
print("CWD now:", os.getcwd())  # should end with Image_Processing_Assignment_1

CWD now: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1


In [2]:
from pathlib import Path
import os

print("CWD:", os.getcwd())
raw = Path("data/raw")
print("data/raw exists? ", raw.exists())
print("Files I can see in data/raw:")
for p in raw.glob("*"):
    print(" -", p.name)

# Try to detect the shark file automatically (case/space/extension tolerant)
candidates = []
patterns = [
    "Shark 2.*", "Shark_2.*", "shark 2.*", "shark_2.*",
    "Shark2.*", "shark2.*", "Shark 2*"
]
ex = [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]

for pat in patterns:
    for p in raw.glob(pat):
        if p.suffix in ex:
            candidates.append(p)

if candidates:
    INPUT_PATH = sorted(candidates, key=lambda p: p.name.lower())[0]
    print("Auto-selected:", INPUT_PATH)
else:
    # Fallback: keep whatever you already had, but warn
    INPUT_PATH = Path("data/raw/Shark 2.PNG")
    print("No candidates found. Expecting:", INPUT_PATH)

CWD: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1
data/raw exists?  True
Files I can see in data/raw:
 - snow_cat.jpg
 - Shark 2.PNG
 - Shark 1.PNG
Auto-selected: data/raw/Shark 2.PNG


In [24]:
# ===== Single-pass fish enhancement & isolation (no CLAHE) =====
from pathlib import Path
import cv2 as cv
import numpy as np
import os

# ---------- user paths (edit these) ----------
# If your notebook is inside notebooks/, jump to project root:
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(str(cwd.parent))

INPUT_PATH  = Path("data/raw/snow_cat.jpg")       # e.g., "data/raw/Shark 1.PNG" or any test image
OUTPUT_PATH = Path("outputs/cat_final.png")     # output image
# ---------------------------------------------

def gray_world_wb(img):
    """Simple gray-world white balance: scale each B,G,R so their means match."""
    img = img.astype(np.float32)
    means = img.reshape(-1,3).mean(axis=0) + 1e-6
    scale = means.mean() / means
    wb = np.clip(img * scale, 0, 255).astype(np.uint8)
    return wb

def process_image(input_path: Path, output_path: Path):
    I = cv.imread(str(input_path), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(f"Input image not found: {input_path}")
    h, w = I.shape[:2]

    # 1) White balance (gray-world) + 2) light denoise (median)
    #I_wb  = gray_world_wb(I)
    den   = cv.medianBlur(I, 5)  # simpler than bilateral; preserves edges reasonably well

    # 3) Simple gamma correction (brighten mid-tones a little)
    gamma = 1.10  # 1.0 = no change; <1 brightens, >1 darkens (here we want a gentle lift)
    invg  = 1.0 / gamma
    # build a tiny 256-LUT for speed
    lut = np.array([((i/255.0)**invg)*255.0 for i in range(256)], dtype=np.uint8)
    enh = cv.LUT(den, lut)

    # 4) Segmentation: Otsu on LAB-b (with small fallback using HSV-S)
    lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB)
    b   = lab[:,:,2]
    _, th_b = cv.threshold(b, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    hsv = cv.cvtColor(enh, cv.COLOR_BGR2HSV)
    S   = hsv[:,:,1]
    _, th_s = cv.threshold(S, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    # start with b only; fallback if too small/too large
    mask = th_b.copy()
    area = (mask.sum()/255) / mask.size
    if area < 0.02:                # too small -> relax with OR
        mask = cv.bitwise_or(th_b, th_s)
    elif area > 0.90:              # too big -> tighten using inverted S
        _, s_inv = cv.threshold(255 - S, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
        mask = cv.bitwise_and(th_b, s_inv)
    # 5) Morphology (gentle) + largest component
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5,5))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=1)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN,  k, iterations=1)
    
    area_ratio = (mask.sum()/255) / mask.size
    print(f"[mask] area={area_ratio:.4f}")
    
    num, labels = cv.connectedComponents(mask)
    if num > 1:
        areas = np.array([(labels==i).sum() for i in range(1, num)])
        idx   = 1 + int(np.argmax(areas))
        big   = (labels==idx).astype(np.uint8)*255
        # if even the largest is tiny, fall back to th_b (more conservative)
        mask  = big if areas.max()/mask.size >= 0.01 else th_b
    
    # 6) Auto-rotate with minAreaRect and tight crop  (← NOT indented under the block above)
    cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt   = max(cnts, key=cv.contourArea)
        rect  = cv.minAreaRect(cnt)
        angle = rect[-1]
        (rw, rh) = rect[1]
        if rw < rh:        # make long side horizontal
            angle += 90
        if angle < -45:
            angle += 90
    
        Mrot  = cv.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        I_rot = cv.warpAffine(enh, Mrot, (w, h), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
        m_rot = cv.warpAffine(mask, Mrot, (w, h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)
    
        # Transform the fish contour for a stable bbox
        cnt_f  = cnt.reshape(-1,1,2).astype(np.float32)
        cnt_rt = cv.transform(cnt_f, Mrot).astype(np.int32)
        x,y,ww,hh = cv.boundingRect(cnt_rt)
        print(f"[rotate] angle={angle:.2f}  bbox={ww}x{hh}  image={w}x{h}")
    
        # Avoid razor-thin crops
        min_frac = 0.2
        if ww < int(min_frac*w) or hh < int(min_frac*h):
            print("[crop] bbox too thin → skipping crop")
            crop, mcrp = I_rot, m_rot
        else:
            crop = I_rot[y:y+hh, x:x+ww]
            mcrp = m_rot[y:y+hh, x:x+ww]
    else:
        print("[rotate] no contour → use enhanced image")
        crop, mcrp = enh, mask
    
            



        
    # 7) Replace background with a solid, scene-matched colour
    inv = (mcrp==0).astype(np.uint8)*255
    border = np.zeros_like(inv)
    border[:10,:]  = 255; border[-10:,:] = 255
    border[:,:10]  = 255; border[:,-10:] = 255

    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1) > 0)]
    color   = tuple(map(int, (np.median(samples,0) if len(samples) else np.array([210,220,230]))))
    canvas  = np.full_like(crop, color, dtype=np.uint8)

    final = cv.add(cv.bitwise_and(crop,  crop,  mask=mcrp),
                   cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mcrp)))

   
    

    output_path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv.imwrite(str(output_path), final)
    print("[OK]" if ok else "[ERROR] Save failed:", output_path.resolve())

process_image(INPUT_PATH, OUTPUT_PATH)

[mask] area=0.0216
[rotate] angle=23.03  bbox=160x137  image=960x639
[crop] bbox too thin → skipping crop
[OK] /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/outputs/cat_final.png


In [2]:
# ===== Single-pass fish enhancement & isolation (auto-picks mask for sharks OR snow-cat) =====
from pathlib import Path
import cv2 as cv
import numpy as np
import os

# --- ensure we run from project root (so data/raw exists) ---
here = Path.cwd()
if not (here / "data" / "raw").exists():
    if (here.parent / "data" / "raw").exists():
        os.chdir(str(here.parent))

# ---- user paths (edit these two only) ----
INPUT_PATH  = Path("data/raw/Shark 1.PNG")         # or "data/raw/Shark 1.PNG" or "data/raw/snow_cat.jpg"
OUTPUT_PATH = Path("outputs/fish_final.png")
# ------------------------------------------

def enhance(img):
    """Mild edge-preserving clean + local contrast + light sharpen."""
    den  = cv.bilateralFilter(img, d=7, sigmaColor=60, sigmaSpace=60)
    LAB  = cv.cvtColor(den, cv.COLOR_BGR2LAB)
    L,A,B = cv.split(LAB)
    Lc   = cv.createCLAHE(clipLimit=2.5, tileGridSize=(8,8)).apply(L)
    enh  = cv.cvtColor(cv.merge([Lc, A, B]), cv.COLOR_LAB2BGR)
    blur = cv.GaussianBlur(enh, (0,0), 1.0)
    enh  = cv.addWeighted(enh, 1.3, blur, -0.3, 0)
    return enh

def otsu(img):
    return cv.threshold(img, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)[1]

def pick_mask(enh):
    """Build two very simple masks and pick the better one by edge score + sane area."""
    hsv = cv.cvtColor(enh, cv.COLOR_BGR2HSV); S = hsv[:,:,1]
    lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB); a = lab[:,:,1]; b = lab[:,:,2]

    # Underwater (sharks): S AND b
    m_shark = cv.bitwise_and(otsu(S), otsu(b))

    # Snow/neutral (cat): inverted S AND a
    Sinv = 255 - S
    m_cat = cv.bitwise_and(otsu(Sinv), otsu(a))

    H, W = S.shape
    edges = cv.Canny(cv.cvtColor(enh, cv.COLOR_BGR2GRAY), 50, 150)

    def score(mask):
        area = (mask.sum() / 255.0) / (H*W + 1e-6)         # fraction of image
        if area < 0.01 or area > 0.9:                      # reject silly masks
            return -1.0, area
        edge_density = (cv.bitwise_and(edges, edges, mask=mask).sum() / 255.0) / (mask.sum()/255.0 + 1e-6)
        return edge_density, area

    s1, a1 = score(m_shark)
    s2, a2 = score(m_cat)

    mask = m_shark if s1 >= s2 else m_cat
    # small tidy to avoid specks/holes but keep detail
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5,5))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=1)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN,  k, iterations=1)

    # keep largest component; if even that is tiny, fall back to the other mask
    num, labels = cv.connectedComponents(mask)
    if num > 1:
        sizes = np.array([(labels==i).sum() for i in range(1, num)])
        idx   = 1 + int(np.argmax(sizes))
        big   = (labels==idx).astype(np.uint8) * 255
        if sizes.max() / (H*W) >= 0.01:
            mask = big

    return mask

def rotate_and_crop(enh, mask):
    h, w = mask.shape
    cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return enh, mask
    cnt   = max(cnts, key=cv.contourArea)
    rect  = cv.minAreaRect(cnt)
    angle = rect[-1]
    rw, rh = rect[1]
    if rw < rh: angle += 90
    if angle < -45: angle += 90

    Mrot  = cv.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    I_rot = cv.warpAffine(enh,  Mrot, (w, h), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
    m_rot = cv.warpAffine(mask, Mrot, (w, h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)

    # crop using transformed contour for stability
    cnt_f  = cnt.reshape(-1,1,2).astype(np.float32)
    cnt_rt = cv.transform(cnt_f, Mrot).astype(np.int32)
    x,y,ww,hh = cv.boundingRect(cnt_rt)

    # guard against silly slivers
    if ww < int(0.2*w) or hh < int(0.2*h):
        return I_rot, m_rot
    return I_rot[y:y+hh, x:x+ww], m_rot[y:y+hh, x:x+ww]

def compose_on_solid(crop, mask_crop):
    inv = (mask_crop==0).astype(np.uint8)*255
    border = np.zeros_like(inv)
    border[:10,:]  = 255; border[-10:,:] = 255
    border[:,:10]  = 255; border[:,-10:] = 255

    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1) > 0)]
    color   = tuple(map(int, (np.median(samples,0) if len(samples) else np.array([210,220,230]))))
    canvas  = np.full_like(crop, color, dtype=np.uint8)

    final = cv.add(cv.bitwise_and(crop,  crop,  mask=mask_crop),
                   cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mask_crop)))
    return final

def process_image(input_path: Path, output_path: Path):
    I = cv.imread(str(input_path), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(f"Input image not found: {input_path}")
    enh  = enhance(I)
    mask = pick_mask(enh)

    # safety: if mask is almost empty, just write the enhanced image
    if (mask.sum()/255.0) / mask.size < 0.005:
        out = enh
    else:
        crop, mcrp = rotate_and_crop(enh, mask)
        # safety #2: if still tiny, bail out to enhanced
        if (mcrp.sum()/255.0) / mcrp.size < 0.005:
            out = enh
        else:
            out = compose_on_solid(crop, mcrp)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv.imwrite(str(output_path), out)
    print(f"[OK] Wrote: {output_path}")

process_image(INPUT_PATH, OUTPUT_PATH)


[OK] Wrote: outputs/fish_final.png


In [15]:
# ==== Single-pass enhancement & isolation (Transformation-focused) ====
from pathlib import Path
import cv2 as cv
import numpy as np
import os

# --- run from project root if notebook lives in notebooks/ or outputs/ ---
here = Path.cwd()
if not (here / "data" / "raw").exists():
    for _ in range(4):
        if (here.parent / "data" / "raw").exists():
            os.chdir(str(here.parent)); break
        here = here.parent
print("CWD:", Path.cwd())

# -------- user paths (edit only these two) --------
INPUT_PATH  = Path("data/raw/Shark 1.PNG")        # e.g. Shark 1.PNG or snow_cat.jpg
OUTPUT_PATH = Path("outputs/fish_final.png")      # single final image
# -------------------------------------------------

def process_image(input_path: Path, output_path: Path):
    I = cv.imread(str(input_path), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(f"Input not found: {input_path}")
    h, w = I.shape[:2]

    # 1) Global L equalisation (simple, from notes)
    LAB  = cv.cvtColor(I, cv.COLOR_BGR2LAB)
    L,A,B = cv.split(LAB)
    L_eq  = cv.equalizeHist(L)
    I_eq  = cv.cvtColor(cv.merge([L_eq, A, B]), cv.COLOR_LAB2BGR)  # <- defines I_eq
    
    # 2) Light unsharp (gentle)
    blur = cv.GaussianBlur(I_eq, (0,0), 1.0)
    enh  = cv.addWeighted(I_eq, 1.12, blur, -0.12, 0)
    
    # 3) Tiny gamma lift
    gamma = 1.04
    lut   = np.array([((i/255.0)**(1.0/gamma))*255.0 for i in range(256)], dtype=np.uint8)
    enh   = cv.LUT(enh, lut)
    
    # Prepare grayscale once for edges/thresholds later
    gray  = cv.cvtColor(enh, cv.COLOR_BGR2GRAY)


    # 4) Segmentation: try several simple Otsu masks, score by edge density + sane area
    lab2 = cv.cvtColor(enh, cv.COLOR_BGR2LAB); Lc, Ac, Bc = cv.split(lab2)
    hsv  = cv.cvtColor(enh, cv.COLOR_BGR2HSV); S = hsv[:,:,1]
    
    _, th_b      = cv.threshold(Bc,       0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # LAB-b
    _, th_a      = cv.threshold(Ac,       0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # LAB-a
    _, s_inv     = cv.threshold(255 - S,  0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # inverted S
    _, th_gray   = cv.threshold(gray,     0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # grayscale
    th_gray_inv  = cv.bitwise_not(th_gray)

    b_and_s = cv.bitwise_and(th_b, s_inv) 
    cands = {"lab_b": th_b, "lab_a": th_a, "s_inv": s_inv, "b_and_s": b_and_s, "gray": th_gray, "gray_inv": th_gray_inv}
    
    def area_ratio(m): 
        return (m.sum()/255) / m.size

    edges_for_score = cv.Canny(cv.medianBlur(gray, 3), 50, 150)
    edges_bool = (edges_for_score > 0).astype(np.uint8)
    
    def border_touch_ratio(m):
        top = (m[0,:]   > 0).sum()
        bot = (m[-1,:]  > 0).sum()
        lef = (m[:,0]   > 0).sum()
        rig = (m[:,-1]  > 0).sum()
        perim = 2*(m.shape[0] + m.shape[1])
        return (top+bot+lef+rig) / (perim + 1e-6)
    
    def aspect_penalty(m):
        cnts,_ = cv.findContours(m, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        if not cnts: 
            return 0.4
        cnt = max(cnts, key=cv.contourArea)
        x,y,w,h = cv.boundingRect(cnt)
        r = w / max(1,h)
        # prefer elongated shapes (roughly 1.5–6)
        return 0.0 if 1.5 <= r <= 6.0 else 0.25
    
    def score_mask(m):
        ar = area_ratio(m)
        # edge density
        m_bool = (m > 0).astype(np.uint8)
        edge_in = (edges_for_score & m_bool).sum()
        pix_in  = m_bool.sum() + 1e-6
        edge_density = edge_in / pix_in
    
        # base area penalty (aim ~0.15)
        penalty = abs(ar - 0.15)
        if ar < 0.02 or ar > 0.60:
            penalty += 0.6
    
        # extra penalties
        penalty += aspect_penalty(m)
        if border_touch_ratio(m) > 0.15:
            penalty += 0.3
    
        return penalty + (1.0 - edge_density)   # lower is better
    
    best_key = min(cands, key=lambda k: score_mask(cands[k]))
    mask = cands[best_key].copy()
    print(f"[seg] picked {best_key}, area={area_ratio(mask):.4f}")
    
    # 5) Morphology (gentle) + largest component (unchanged)
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5,5))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=1)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN,  k, iterations=1)
    
    num, labels = cv.connectedComponents(mask)
    if num > 1:
        areas = np.array([(labels==i).sum() for i in range(1, num)])
        idx   = 1 + int(np.argmax(areas))
        big   = (labels==idx).astype(np.uint8)*255
        # if even the largest is tiny, keep the original best mask (still conservative)
        mask  = big if areas.max()/mask.size >= 0.01 else mask

    # 6) Auto-rotate and tight crop using transformed contour  [Transformation: Rotating / Adjustments]
    cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt   = max(cnts, key=cv.contourArea)
        rect  = cv.minAreaRect(cnt)
        angle = rect[-1]
        (rw, rh) = rect[1]
        if rw < rh: angle += 90
        if angle < -45: angle += 90

        Mrot  = cv.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        I_rot = cv.warpAffine(enh,  Mrot, (w, h), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
        m_rot = cv.warpAffine(mask, Mrot, (w, h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)

        # transform the contour for a stable bbox
        cnt_f  = cnt.reshape(-1,1,2).astype(np.float32)
        cnt_rt = cv.transform(cnt_f, Mrot).astype(np.int32)
        x,y,ww,hh = cv.boundingRect(cnt_rt)

        # avoid razor-thin crops
        min_frac = 0.15
        if ww < int(min_frac*w) or hh < int(min_frac*h):
            crop, mcrp = I_rot, m_rot
        else:
            crop = I_rot[y:y+hh, x:x+ww]
            mcrp = m_rot[y:y+hh, x:x+ww]
    else:
        crop, mcrp = enh, mask

    # 7) Minimal background: solid colour from border median  [simple masking from notes]
    inv = (mcrp==0).astype(np.uint8)*255
    border = np.zeros_like(inv); border[:10,:]=255; border[-10:,:]=255; border[:,:10]=255; border[:,-10:]=255
    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1) > 0)]
    color   = tuple(map(int, (np.median(samples,0) if len(samples) else np.array([210,220,230]))))
    canvas  = np.full_like(crop, color, dtype=np.uint8)

    final = cv.add(cv.bitwise_and(crop,  crop,  mask=mcrp),
                   cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mcrp)))

    # safety: if mask basically empty, return enhanced image (still valid “one output”)
    if (mcrp.sum()/255)/mcrp.size < 0.005:
        final = enh

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv.imwrite(str(output_path), final)
    print(f"[OK] wrote: {output_path.resolve()}")

process_image(INPUT_PATH, OUTPUT_PATH)


CWD: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1
[seg] picked b_and_s, area=0.2250
[OK] wrote: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/outputs/fish_final.png


In [34]:
# ==== Single-pass: fusion enhancement + simple segmentation → one output ====
from pathlib import Path
import cv2 as cv
import numpy as np
import os

# -- ensure we run from project root (so data/raw exists) --
here = Path.cwd()
if not (here/"data"/"raw").exists():
    for _ in range(4):
        if (here.parent/"data"/"raw").exists():
            os.chdir(str(here.parent)); break
        here = here.parent
print("CWD:", Path.cwd())

# ---- user paths ----
INPUT_PATH  = Path("data/raw/Shark 2.PNG")  # change to Shark 2.PNG or your cat
OUTPUT_PATH = Path("outputs/fish_final.png")

# ---------- helpers ----------
def gray_world_wb(img, clamp=(0.85, 1.20)):
    """Gray-world white balance with gentle clamping to avoid color casts."""
    f   = img.astype(np.float32)
    mu  = f.reshape(-1,3).mean(axis=0) + 1e-6
    scl = mu.mean()/mu
    scl = np.clip(scl, clamp[0], clamp[1])
    out = np.clip(f*scl, 0, 255).astype(np.uint8)
    return out

def branch_a(img):
    """WB → gamma → CLAHE (on L)"""
    wb  = gray_world_wb(img)
    # gamma (mild)
    gamma = 1.1
    lut   = np.array([((i/255.0)**(1.0/gamma))*255.0 for i in range(256)], dtype=np.uint8)
    g    = cv.LUT(wb, lut)
    # CLAHE on L
    lab  = cv.cvtColor(g, cv.COLOR_BGR2LAB)
    L,A,B = cv.split(lab)
    L2   = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(L)
    out  = cv.cvtColor(cv.merge([L2,A,B]), cv.COLOR_LAB2BGR)
    return out

def branch_b(img):
    """WB → mild unsharp → global HE (on L)"""
    wb   = gray_world_wb(img)
    blur = cv.GaussianBlur(wb, (0,0), 1.0)
    sh   = cv.addWeighted(wb, 1.15, blur, -0.15, 0)
    lab  = cv.cvtColor(sh, cv.COLOR_BGR2LAB)
    L,A,B = cv.split(lab)
    L2   = cv.equalizeHist(L)
    out  = cv.cvtColor(cv.merge([L2,A,B]), cv.COLOR_LAB2BGR)
    return out

def fuse_mertens(img1, img2):
    """Exposure fusion (multiscale pyramid under the hood)."""
    m = cv.createMergeMertens(contrast_weight=1.0, saturation_weight=0.6, exposure_weight=0.6)
    f = m.process([img1.astype(np.float32)/255.0, img2.astype(np.float32)/255.0])
    out = np.clip(f*255.0, 0, 255).astype(np.uint8)
    return out

def pick_mask(enh):
    """Simple, robust mask: Otsu on LAB-b and (inverted) HSV-S, plus grayscale fallback."""
    lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB); L,A,B = cv.split(lab)
    hsv = cv.cvtColor(enh, cv.COLOR_BGR2HSV); S     = hsv[:,:,1]
    gray = cv.cvtColor(enh, cv.COLOR_BGR2GRAY)

    _, th_b    = cv.threshold(B,       0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # blue/yellow split
    _, s_inv   = cv.threshold(255-S,   0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # prefer higher sat
    b_and_s    = cv.bitwise_and(th_b, s_inv)

    _, th_gray = cv.threshold(gray,    0, 255, cv.THRESH_BINARY+cv.THRESH_OTSU)  # fallback (non-underwater)
    th_gray_inv = cv.bitwise_not(th_gray)

    cands = {
        "b_and_s": b_and_s,       # usually best for fish vs sea
        "lab_b"  : th_b,
        "s_inv"  : s_inv,
        "gray"   : th_gray,
        "gray_inv": th_gray_inv,
    }
    def area_ratio(m): return (m.sum()/255)/m.size

    # choose by area (prefer around 15%, reject <1.5% or >70%)
    def score(m):
        ar = area_ratio(m)
        pen = abs(ar-0.15)
        if ar < 0.015 or ar > 0.70: pen += 0.7
        return pen

    best_key = min(cands, key=lambda k: score(cands[k]))
    mask = cands[best_key].copy()
    print(f"[seg] picked {best_key}, area={area_ratio(mask):.4f}")
    return mask

def clean_biggest(mask):
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5,5))
    m = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=1)
    m = cv.morphologyEx(m,    cv.MORPH_OPEN,  k, iterations=1)
    n, lab = cv.connectedComponents(m)
    if n>1:
        areas = np.array([(lab==i).sum() for i in range(1,n)])
        i     = 1 + int(np.argmax(areas))
        m     = (lab==i).astype(np.uint8)*255
    return m

def rotate_crop(enh, mask):
    h,w = mask.shape
    cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return enh, mask
    cnt   = max(cnts, key=cv.contourArea)
    rect  = cv.minAreaRect(cnt)
    angle = rect[-1]; (rw,rh) = rect[1]
    if rw<rh: angle += 90
    if angle < -45: angle += 90

    M    = cv.getRotationMatrix2D((w//2,h//2), angle, 1.0)
    eR   = cv.warpAffine(enh,  M, (w,h), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
    mR   = cv.warpAffine(mask, M, (w,h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)

    # stable bbox from transformed contour
    cnt_t  = cv.transform(cnt.reshape(-1,1,2).astype(np.float32), M).astype(np.int32)
    x,y,ww,hh = cv.boundingRect(cnt_t)
    # avoid silly thin crops
    if ww < 0.15*w or hh < 0.15*h:
        return eR, mR
    return eR[y:y+hh, x:x+ww], mR[y:y+hh, x:x+ww]

def compose_min_bg(crop, mcrp):
    inv = (mcrp==0).astype(np.uint8)*255
    border = np.zeros_like(inv)
    border[:10,:]=255; border[-10:,:]=255; border[:,:10]=255; border[:,-10:]=255
    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1)>0)]
    color   = tuple(map(int, (np.median(samples,0) if len(samples) else np.array([210,220,230]))))
    canvas  = np.full_like(crop, color, dtype=np.uint8)
    out = cv.add(cv.bitwise_and(crop,  crop,  mask=mcrp),
                 cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mcrp)))
    return out

# ---------- main ----------
def process_image(inp, outp):
    I = cv.imread(str(inp), cv.IMREAD_COLOR)
    if I is None: raise FileNotFoundError(f"Input not found: {inp}")

    # (1) two branches
    A = branch_a(I)
    B = branch_b(I)

    # (2) multiscale fusion (Mertens)
    F = fuse_mertens(A, B)

    # (3) segmentation (simple + fallback)
    mask = pick_mask(F)
    mask = clean_biggest(mask)

    # if mask ended tiny, just return the enhanced image (still one output)
    if (mask.sum()/255)/mask.size < 0.005:
        out = F
    else:
        # (4) auto-rotate, crop and compose minimal background
        crop, mcrp = rotate_crop(F, mask)
        out = compose_min_bg(crop, mcrp)

    outp.parent.mkdir(parents=True, exist_ok=True)
    cv.imwrite(str(outp), out)
    print(f"[OK] wrote: {outp.resolve()}")

process_image(INPUT_PATH, OUTPUT_PATH)


CWD: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1
[seg] picked gray, area=0.0000
[OK] wrote: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/outputs/fish_final.png


In [42]:
# ==== One-input ➜ one-output: simple enhancement + simple segmentation + auto rotate/crop ====
from pathlib import Path
import cv2 as cv
import numpy as np
import os

# --- ensure we run from project root so data/raw exists ---
here = Path.cwd()
if not (here/"data"/"raw").exists():
    for _ in range(4):
        if (here.parent/"data"/"raw").exists():
            os.chdir(str(here.parent)); break
        here = here.parent
print("CWD:", Path.cwd())

# --- user paths (edit INPUT_PATH to test another image) ---
INPUT_PATH  = Path("data/raw/Shark 2.PNG")
#INPUT_PATH  = Path("data/raw/snow_cat.jpg")
OUTPUT_PATH = Path("outputs/fish_final.png")

# ---------------- helpers ----------------
def pick_mask(enh):
        # scene stats
        hsv  = cv.cvtColor(enh, cv.COLOR_BGR2HSV)
        S    = hsv[:,:,1]
        gray = cv.cvtColor(enh, cv.COLOR_BGR2GRAY)
        sat_mean = float(S.mean())
        contrast = float(gray.std())
    
        # ---- colour masks (underwater, Shark 1)
        lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB); _,_,b = cv.split(lab)
        _, th_b  = cv.threshold(b,       0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
        _, s_inv = cv.threshold(255 - S, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
        mask_color_and = cv.bitwise_and(th_b, s_inv)
        mask_color_or  = cv.bitwise_or(th_b,  s_inv)
    
        # ---- silhouette masks (Shark 2: dark-on-bright)
        g_blur  = cv.GaussianBlur(gray, (5,5), 0)
        inv     = cv.normalize(255 - g_blur, None, 0, 255, cv.NORM_MINMAX)
        mask_ad = cv.adaptiveThreshold(inv, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv.THRESH_BINARY, 51, -3)   # slightly looser
        _, mask_otsu_inv = cv.threshold(inv, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    
        # ---- NEW: edge-fill mask (cat / normal photos)
        edges = cv.Canny(gray, 50, 150)
        k5    = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5,5))
        edge_d = cv.dilate(edges, k5, iterations=2)              # thicken edges
        cnts,_ = cv.findContours(edge_d, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        mask_edge = np.zeros_like(gray, np.uint8)
        if cnts:
            cnt = max(cnts, key=cv.contourArea)
            cv.drawContours(mask_edge, [cnt], -1, 255, thickness=cv.FILLED)  # fill inside edge
            # combine with Otsu on gray to avoid spill into background
            _, th_gray = cv.threshold(gray, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
            mask_edge = cv.bitwise_and(mask_edge, th_gray)
    
        def area(m): return (m.sum()/255)/m.size if m is not None else 0.0
    
        # ---- choose mode (simple rules)
        if (sat_mean < 28 and contrast > 30):            # low sat but decent contrast → normal photo
            m = mask_edge if area(mask_edge) >= 0.02 else mask_otsu_inv
            mode = "edge-fill"
        elif (sat_mean < 35) or (contrast < 22):         # silhouette
            m1 = mask_ad if 0.02 <= area(mask_ad) <= 0.70 else mask_otsu_inv
            m  = m1 if 0.02 <= area(m1) <= 0.70 else mask_color_or
            mode = "silhouette"
        else:                                            # colour underwater
            m  = mask_color_and if area(mask_color_and) >= 0.03 else mask_color_or
            mode = "colour"
    
        print(f"[seg] mode={mode}  sat_mean={sat_mean:.1f}  gray_std={contrast:.1f}  area={area(m):.4f}")
        return m, mode
    
def clear_border(m, pad=10):
        m = m.copy()
        m[:pad,:]=0; m[-pad:,:]=0; m[:,:pad]=0; m[:,-pad:]=0
        return m

# ---------------- main pipeline ----------------
def process_image(inp: Path, outp: Path):
    I = cv.imread(str(inp), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(f"Input not found: {inp}")
    h, w = I.shape[:2]

    # (1) Enhancement: bilateral → CLAHE on L → light unsharp
    den  = cv.bilateralFilter(I, d=7, sigmaColor=75, sigmaSpace=75)
    LAB  = cv.cvtColor(den, cv.COLOR_BGR2LAB)
    L,A,B = cv.split(LAB)
    L2   = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(L)
    enh  = cv.cvtColor(cv.merge([L2, A, B]), cv.COLOR_LAB2BGR)
    blur = cv.GaussianBlur(enh, (0,0), 1.0)
    enh  = cv.addWeighted(enh, 1.20, blur, -0.20, 0)

    # (2) Segmentation
    mask, mode = pick_mask(enh)
    mask = clear_border(mask, pad=10)

    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5,5))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=1)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN,  k, iterations=1)

    # keep largest component
    num, labels = cv.connectedComponents(mask)
    if num > 1:
        areas = np.array([(labels==i).sum() for i in range(1, num)])
        idx   = 1 + int(np.argmax(areas))
        mask  = (labels==idx).astype(np.uint8)*255


    # ---- THICKEN & FILL THE MASK (fix skeleton look) ----
    # 1) close with a bigger kernel to bridge gaps
    k_thick = cv.getStructuringElement(cv.MORPH_ELLIPSE, (9,9))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k_thick, iterations=2)
    
    # 2) fill interior holes of the largest component
    cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if cnts:
        hull = cv.convexHull(max(cnts, key=cv.contourArea))
        filled = np.zeros_like(mask)
        cv.drawContours(filled, [hull], -1, 255, thickness=cv.FILLED)
        # prefer hull fill only if it doesn't explode the size (> 3× bigger means sea leak)
        old_area = (mask.sum()/255)
        new_area = (filled.sum()/255)
        mask = filled if new_area <= 3.0*old_area else mask
    
    # 3) small smooth & re-threshold to tidy edges
    mask_blur = cv.GaussianBlur(mask, (5,5), 0)
    _, mask = cv.threshold(mask_blur, 127, 255, cv.THRESH_BINARY)


    

    
    # small-mask grow
    ar = (mask.sum()/255)/mask.size
    if 0.01 <= ar < 0.08:
        mask = cv.dilate(mask, cv.getStructuringElement(cv.MORPH_ELLIPSE,(7,7)), iterations=2)
       
        ar     = (mask.sum()/255)/mask.size
        print(f"[seg] dilated small mask → area={ar:.4f}")

    # fallback: if unusable mask, just save enhanced image
    if ar < 0.005 or ar > 0.98:
        outp.parent.mkdir(parents=True, exist_ok=True)
        cv.imwrite(str(outp), enh)
        print("[WARN] mask unusable → wrote enhanced image only:", outp.resolve())
        return

    # ---- CROP ----
    if mode == "edge-fill":
        # No rotate: axis-aligned bbox works better for normal photos
        cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        cnt    = max(cnts, key=cv.contourArea)
        x,y,ww,hh = cv.boundingRect(cnt)
        pad = int(0.18 * min(mask.shape))    # a bit roomier for ears/tail
        h,w = mask.shape
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(w, x + ww + pad); y1 = min(h, y + hh + pad)
        crop = enh[y0:y1, x0:x1]
        mcrp = mask[y0:y1, x0:x1]
    else:
        # Your existing auto-rotate + rotated-bbox path (for sharks)
        cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        cnt    = max(cnts, key=cv.contourArea)
        rect   = cv.minAreaRect(cnt)
        angle  = rect[-1]; (rw, rh) = rect[1]
        if rw < rh: angle += 90
        if angle < -45: angle += 90
        h,w = mask.shape
        Mrot  = cv.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        I_rot = cv.warpAffine(enh,  Mrot, (w, h), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
        m_rot = cv.warpAffine(mask, Mrot, (w, h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)
        cnt_t = cv.transform(cnt.reshape(-1,1,2).astype(np.float32), Mrot).astype(np.int32)
        x,y,ww,hh = cv.boundingRect(cnt_t)
        pad = int(0.15 * min(w, h))
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(w, x + ww + pad); y1 = min(h, y + hh + pad)
        crop = I_rot[y0:y1, x0:x1]
        mcrp = m_rot[y0:y1, x0:x1]


    # (4) Clean minimal background, softly blur only the background
    inv = (mcrp==0).astype(np.uint8)*255
    border = np.zeros_like(inv)
    border[:10,:]=255; border[-10:,:]=255; border[:,:10]=255; border[:,-10:]=255
    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1) > 0)]
    bg_colour = (210,220,230) if len(samples)==0 else tuple(map(int, np.median(samples,0)))
    canvas = np.full_like(crop, bg_colour, dtype=np.uint8)
    final  = cv.add(cv.bitwise_and(crop,  crop,  mask=mcrp),
                    cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mcrp)))

    # optional: soften only background
    bg_mask = cv.bitwise_not(mcrp)
    bg_blur = cv.bilateralFilter(final, d=7, sigmaColor=60, sigmaSpace=60)
    final   = cv.add(cv.bitwise_and(final, final, mask=mcrp),
                     cv.bitwise_and(bg_blur, bg_blur, mask=bg_mask))

    # (5) Save one output
    outp.parent.mkdir(parents=True, exist_ok=True)
    cv.imwrite(str(outp), final)
    print(f"[OK] wrote: {outp.resolve()}")

process_image(INPUT_PATH, OUTPUT_PATH)


CWD: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1
[seg] mode=silhouette  sat_mean=229.6  gray_std=16.9  area=0.2422
[seg] dilated small mask → area=0.0602
[OK] wrote: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/outputs/fish_final.png


In [55]:
# ==== Simple pipeline (lab style): enhance → threshold → tidy → crop → clean background ====
from pathlib import Path
import cv2 as cv
import numpy as np
import os

# -------- parameters you can tweak --------
INPUT_PATH  = Path("data/raw/Shark 2.PNG")   # change for Shark 2 or the cat
#INPUT_PATH  = Path("data/raw/snow_cat.jpg")
OUTPUT_PATH = Path("outputs/final.png")

MODE   = "underwater"   # "underwater" (LAB-b & saturation)  or  "silhouette" (inverted gray)
ROTATE = True           # False for cat (normal photo); True for sharks
CLAHE_CLIP = 3.0        # 2.0–4.0 are sensible
CLOSE_K    = 5          # morphology kernel size (odd): 5 or 7
CLOSE_IT   = 1          # closing iterations
DILATE_IT  = 2          # grow mask a little if thin (try 1–3)
PAD_FRAC   = 0.15       # crop padding as fraction of min(image size)
# ------------------------------------------

# ensure we run from project root so data/raw exists
here = Path.cwd()
if not (here/"data"/"raw").exists():
    for _ in range(4):
        if (here.parent/"data"/"raw").exists():
            os.chdir(str(here.parent)); break
        here = here.parent
print("CWD:", Path.cwd())

def process_image(inp: Path, outp: Path):
    I = cv.imread(str(inp), cv.IMREAD_COLOR)
    if I is None:
        raise FileNotFoundError(f"Input not found: {inp}")
    H, W = I.shape[:2]

    # 1) Enhancement (all in labs): bilateral → CLAHE(L) → light unsharp
    den = cv.bilateralFilter(I, d=7, sigmaColor=75, sigmaSpace=75)
    L,A,B = cv.split(cv.cvtColor(den, cv.COLOR_BGR2LAB))
    L2 = cv.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8,8)).apply(L)
    enh = cv.cvtColor(cv.merge([L2, A, B]), cv.COLOR_LAB2BGR)
    blur = cv.GaussianBlur(enh, (0,0), 1.0)
    enh  = cv.addWeighted(enh, 1.20, blur, -0.20, 0)  # gentle

    # 2) Segmentation (two very simple options)
    if MODE == "underwater":
        lab = cv.cvtColor(enh, cv.COLOR_BGR2LAB); _,_,b = cv.split(lab)
        hsv = cv.cvtColor(enh, cv.COLOR_BGR2HSV); S    = hsv[:,:,1]
        _, th_b  = cv.threshold(b,       0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
        _, s_inv = cv.threshold(255 - S, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
        mask = cv.bitwise_and(th_b, s_inv)   # strict
        # if too small, relax to OR (still simple)
        area = (mask.sum()/255)/mask.size
        if area < 0.03:
            mask = cv.bitwise_or(th_b, s_inv)
    else:  # MODE == "silhouette"  → works for Shark 2 and the cat
        gray  = cv.cvtColor(enh, cv.COLOR_BGR2GRAY)
        g_blr = cv.GaussianBlur(gray, (5,5), 0)
        inv   = cv.normalize(255 - g_blr, None, 0, 255, cv.NORM_MINMAX)
        _, mask = cv.threshold(inv, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    # 3) Tidy: close → open → (optionally) dilate a little
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (CLOSE_K, CLOSE_K))
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k, iterations=CLOSE_IT)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN,  k, iterations=1)
    if DILATE_IT > 0:
        mask = cv.dilate(mask, k, iterations=DILATE_IT)

    # keep largest component
    num, labels = cv.connectedComponents(mask)
    if num > 1:
        areas = [(labels==i).sum() for i in range(1, num)]
        idx   = 1 + int(np.argmax(areas))
        mask  = (labels==idx).astype(np.uint8)*255

    # fallback: if mask is still basically empty/full → just save enhanced image
    area = (mask.sum()/255)/mask.size
    if area < 0.005 or area > 0.98:
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        cv.imwrite(str(outp), enh)
        print("[WARN] mask unusable → wrote enhanced image only:", outp.resolve())
        return

    # 4) Crop (rotate for sharks, axis-aligned for cat)
    if ROTATE:
        cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        cnt = max(cnts, key=cv.contourArea)
        rect  = cv.minAreaRect(cnt)
        angle = rect[-1]; (rw, rh) = rect[1]
        if rw < rh: angle += 90
        if angle < -45: angle += 90
        M = cv.getRotationMatrix2D((W//2, H//2), angle, 1.0)
        I_rot = cv.warpAffine(enh,  M, (W, H), flags=cv.INTER_LINEAR,  borderMode=cv.BORDER_REFLECT)
        m_rot = cv.warpAffine(mask, M, (W, H), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_CONSTANT)
        cnt_t = cv.transform(cnt.reshape(-1,1,2).astype(np.float32), M).astype(np.int32)
        x,y,w,h = cv.boundingRect(cnt_t)
        pad = int(PAD_FRAC * min(W, H))
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(W, x + w + pad); y1 = min(H, y + h + pad)
        crop, mcrp = I_rot[y0:y1, x0:x1], m_rot[y0:y1, x0:x1]
    else:
        cnts,_ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        cnt = max(cnts, key=cv.contourArea)
        x,y,w,h = cv.boundingRect(cnt)
        pad = int(PAD_FRAC * min(W, H))
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(W, x + w + pad); y1 = min(H, y + h + pad)
        crop, mcrp = enh[y0:y1, x0:x1], mask[y0:y1, x0:x1]

    # 5) Clean, minimal background (solid colour from border)
    inv = (mcrp==0).astype(np.uint8)*255
    border = np.zeros_like(inv); border[:10,:]=255; border[-10:,:]=255; border[:,:10]=255; border[:,-10:]=255
    samples = cv.bitwise_and(crop, crop, mask=cv.bitwise_and(inv, border)).reshape(-1,3)
    samples = samples[(samples.sum(1) > 0)]
    bg_col  = (210,220,230) if len(samples)==0 else tuple(map(int, np.median(samples,0)))
    canvas  = np.full_like(crop, bg_col, dtype=np.uint8)
    final   = cv.add(cv.bitwise_and(crop,  crop,  mask=mcrp),
                     cv.bitwise_and(canvas, canvas, mask=cv.bitwise_not(mcrp)))

    # save one output
    outp.parent.mkdir(parents=True, exist_ok=True)
    cv.imwrite(str(outp), final)
    print(f"[OK] wrote: {outp.resolve()}  |  mode={MODE}, rotate={ROTATE}, area={area:.3f}")

process_image(INPUT_PATH, OUTPUT_PATH)


CWD: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1
[OK] wrote: /Users/aaronbaggot/Desktop/Image_Processing_Assignment_1/outputs/final.png  |  mode=underwater, rotate=True, area=0.619
